In [31]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
VOCAB_URL = "https://raw.githubusercontent.com/shuzi/insuranceQA/master/V2/vocabulary"
vocab_df = pd.read_csv(
    VOCAB_URL, sep="\t", header=None, names=["idx", "word"],
    quoting=3, dtype=str, keep_default_na=False,
)
vocab = dict(zip(vocab_df["idx"], vocab_df["word"]))

print("total vocab entries:", len(vocab))
print(vocab_df["word"].apply(type).value_counts())  # should show ONLY <class 'str'> now
print("\nidx_12025 ->", repr(vocab["idx_12025"]))  # should be 'None' the string
print("idx_21568 ->", repr(vocab["idx_21568"]))    # should be 'null' the string

print("sample:")
for k in list(vocab)[:5]:
    print(f"  {k} -> {vocab[k]}")


total vocab entries: 68580
word
<class 'str'>    68580
Name: count, dtype: int64

idx_12025 -> 'None'
idx_21568 -> 'null'
sample:
  idx_17904 -> rating/result
  idx_14300 -> considered,
  idx_56809 -> Nonforfeiture
  idx_3159 -> considered.
  idx_65641 -> 3-24-2015


In [19]:
QUESTIONS_URL = "https://raw.githubusercontent.com/shuzi/insuranceQA/master/V2/InsuranceQA.question.anslabel.raw.100.pool.solr.test.encoded.gz"

questions_df = pd.read_csv(
    QUESTIONS_URL, sep="\t", header=None,
    names=["domain", "question_idx", "groundtruth", "pool"],
    compression="gzip", quoting=3,
)
print("total questions:", len(questions_df))
questions_df.head(3)


total questions: 2000


,domain,question_idx,groundtruth,pool
0,life-insurance,idx_1285 idx_65774 idx_862 idx_605 idx_448 idx...,16164 99 26337,15813 3286 22367 21353 4977 6406 24335 16681 2...
1,renters-insurance,idx_1285 idx_1010 idx_999 idx_136 idx_65807,22542 4380,2235 26739 24916 17855 3406 21201 70 19553 220...
2,home-insurance,idx_1010 idx_17002 idx_382 idx_65840 idx_14927...,26439,23486 2424 14974 3344 7712 6220 5346 12474 558...


In [20]:
def decode(idx_string, vocab):
    tokens = idx_string.split()
    return " ".join(vocab.get(tok, f"<UNK:{tok}>") for tok in tokens)

sample_question = questions_df.iloc[0]["question_idx"]
print("encoded:", sample_question)
print("decoded:", decode(sample_question, vocab))


encoded: idx_1285 idx_65774 idx_862 idx_605 idx_448 idx_136 idx_3815 idx_19543 idx_66004
decoded: What Happens When Term Life Insurance Is Paid Up?


In [21]:
ANSWERS_URL = "https://raw.githubusercontent.com/shuzi/insuranceQA/master/V2/InsuranceQA.label2answer.raw.encoded.gz"

answers_df = pd.read_csv(
    ANSWERS_URL, sep="\t", header=None,
    names=["label", "answer_idx"],
    compression="gzip", quoting=3,
)
print("total answers:", len(answers_df))

sample_answer = answers_df.iloc[0]["answer_idx"]
print("\nencoded (truncated):", sample_answer[:150], "...")
print("\ndecoded:", decode(sample_answer, vocab))


total answers: 27413

encoded (truncated): idx_1 idx_2 idx_3 idx_4 idx_5 idx_6 idx_7 idx_8 idx_9 idx_10 idx_11 idx_12 idx_13 idx_14 idx_3 idx_12 idx_15 idx_16 idx_17 idx_8 idx_18 idx_19 idx_20  ...

decoded: Coverage follows the car. Example 1: If you were given a car (loaned) and the car has no insurance, you can buy insurance on the car and your insurance will be primary. Another option, someone helped you to buy a car. For example your credit score isn't good enough to finance, so a friend of yours signed under your loan as a primary payor. You can get insurance under your name and even list your friend on the policy as a loss payee. In this case, we always suggest you get a loan gap coverage: the difference between the car's actual cash value and the amount still owned on it. Example 2: The car you are loaned has insurance. You can buy a policy under your name, list the car on that policy and in case of the accident, your policy will become a secondary or excess. Once the limits of

In [27]:
CORPUS = dict(zip(
    answers_df["label"].astype(str),
    answers_df["answer_idx"].apply(lambda s: decode(s, vocab))
))

print("total real documents in CORPUS:", len(CORPUS))
for k in list(CORPUS)[:2]:
    print(f"\n{k}: {CORPUS[k][:200]}...")


total real documents in CORPUS: 27413

1: Coverage follows the car. Example 1: If you were given a car (loaned) and the car has no insurance, you can buy insurance on the car and your insurance will be primary. Another option, someone helped ...

2: That is a great question! One I'm sorry that you have to ask...I'm sorry for your loss. Each Stat's Department of Insurance will have in its codes the specified time limits for filing claims, beginnin...


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

doc_ids = list(CORPUS.keys())
vectorizer = TfidfVectorizer(stop_words="english")
doc_matrix = vectorizer.fit_transform(CORPUS.values())

def retrieve(query, k=5):
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, doc_matrix)[0]
    ranked = sims.argsort()[::-1][:k]
    return [doc_ids[i] for i in ranked]

# real sanity check: does it find the actual correct answer for a question we know the ground truth for?
test_query = decode(questions_df.iloc[0]["question_idx"], vocab)
true_labels = questions_df.iloc[0]["groundtruth"].split()

print("query:", test_query)
print("true correct label(s):", true_labels)

top5 = retrieve(test_query, k=5)
print("\nretrieved top-5:", top5)
print("any true label in top-5?", any(label in top5 for label in true_labels))


query: What Happens When Term Life Insurance Is Paid Up?
true correct label(s): ['16164', '99', '26337']

retrieved top-5: ['18404', '2700', '11788', '13251', '5121']
any true label in top-5? False


In [32]:
model = SentenceTransformer("all-MiniLM-L6-v2")  # small, fast, well-regarded for exactly this task


doc_ids = list(CORPUS.keys())
doc_texts = list(CORPUS.values())

doc_embeddings = model.encode(doc_texts, show_progress_bar=True, batch_size=64)
print("embeddings shape:", doc_embeddings.shape)  # (27413, 384)

def retrieve_embed(query, k=5):
    q_emb = model.encode([query])[0]
    sims = cosine_similarity([q_emb], doc_embeddings)[0]
    ranked = np.argsort(-sims)[:k]
    return [doc_ids[i] for i in ranked]

# same real check as Step 6 (the TF-IDF one that missed) -- now with embeddings
test_query = decode(questions_df.iloc[0]["question_idx"], vocab)
true_labels = questions_df.iloc[0]["groundtruth"].split()

top5 = retrieve_embed(test_query, k=5)
print("query:", test_query)
print("true labels:", true_labels)
print("retrieved top-5:", top5)
print("any true label in top-5?", any(label in top5 for label in true_labels))


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1087.79it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 429/429 [01:34<00:00,  4.55it/s]


embeddings shape: (27413, 384)
query: What Happens When Term Life Insurance Is Paid Up?
true labels: ['16164', '99', '26337']
retrieved top-5: ['16164', '21000', '16745', '24808', '99']
any true label in top-5? True


In [33]:
import random
random.seed(42)

sample_indices = random.sample(range(len(questions_df)), 8)

GOLDEN_REAL = []
for i in sample_indices:
    row = questions_df.iloc[i]
    GOLDEN_REAL.append({
        "query": decode(row["question_idx"], vocab),
        "relevant": set(row["groundtruth"].split()),
    })

for item in GOLDEN_REAL:
    print(item["query"], "->", item["relevant"])


What Is An Annual Premium For Car Insurance? -> {'2515'}
What Does The Medicare Suffix T Mean? -> {'26868'}
Is Life Insurance An Inheritance? -> {'18229'}
What Is A Good Life Insurance Policy? -> {'8834', '10434'}
Do I Need Homeowners Insurance In A Condo? -> {'4817'}
What Is The Average Cost Of A Whole Life Insurance Policy? -> {'1158', '25897', '16303'}
Can I Get Disability Insurance If Already Pregnant? -> {'22908'}
Who Has The Cheapest Term Life Insurance? -> {'21177', '14822', '4851', '16325', '10107', '721', '8367'}


In [ ]:
def recall_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & relevant) / len(relevant)

def precision_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & relevant) / k

def mrr(retrieved, relevant): # The rank of first relevant result
    for rank, doc in enumerate(retrieved, start=1):
        if doc in relevant:
            return 1 / rank
    return 0.0


K = 3
rows = []
for item in GOLDEN_REAL:
    query = item["query"]
    relevant = item["relevant"]

    tfidf_retrieved = retrieve(query, k=10)
    embed_retrieved = retrieve_embed(query, k=10)

    rows.append({
        "query": query,
        "tfidf_recall@k": recall_at_k(tfidf_retrieved, relevant, K),
        "tfidf_precision@k": precision_at_k(tfidf_retrieved, relevant, K),
        "tfidf_mrr": mrr(tfidf_retrieved, relevant),
        "embed_recall@k": recall_at_k(embed_retrieved, relevant, K),
        "embed_precision@k": precision_at_k(embed_retrieved, relevant, K),
        "embed_mrr": mrr(embed_retrieved, relevant),
    })

results_real = pd.DataFrame(rows)
results_real

'''
Row 0 ("annual premium"): embeddings win outright, TF-IDF total miss, embeddings perfect.
Rows 1, 2: both perfect, tie.
Rows 3, 6: TF-IDF actually beats embeddings, row 6 especially (disability insurance while pregnant): TF-IDF nails the true answer at rank 1, embeddings only gets it to rank 3. My guess: "pregnant" is a rare, highly specific word that TF-IDF weights heavily via IDF, while an embedding model spreads similarity more broadly across the whole "insurance" semantic neighborhood and doesn't necessarily prioritize that one distinctive term as strongly.
Rows 4, 5, 7: both completely miss.
'''

,query,tfidf_recall@k,tfidf_precision@k,tfidf_mrr,embed_recall@k,embed_precision@k,embed_mrr
0,What Is An Annual Premium For Car Insurance?,0.0,0.000000,0.00,1.0,0.333333,1.000000
1,What Does The Medicare Suffix T Mean?,1.0,0.333333,1.00,1.0,0.333333,1.000000
2,Is Life Insurance An Inheritance?,1.0,0.333333,1.00,1.0,0.333333,1.000000
3,What Is A Good Life Insurance Policy?,0.0,0.000000,0.25,0.0,0.000000,0.142857
4,Do I Need Homeowners Insurance In A Condo?,0.0,0.000000,0.00,0.0,0.000000,0.000000
5,What Is The Average Cost Of A Whole Life Insur...,0.0,0.000000,0.00,0.0,0.000000,0.000000
6,Can I Get Disability Insurance If Already Preg...,1.0,0.333333,1.00,1.0,0.333333,0.333333
7,Who Has The Cheapest Term Life Insurance?,0.0,0.000000,0.00,0.0,0.000000,0.000000


$\text{RRF}(d) = \sum \frac{1}{k + \text{rank}_i(d)}$,

In [35]:
def retrieve_hybrid(query, k=5, rrf_k=60):
    tfidf_ranked = retrieve(query, k=20)
    embed_ranked = retrieve_embed(query, k=20)

    scores = {}
    for ranks in [tfidf_ranked, embed_ranked]:
        for rank, doc_id in enumerate(ranks, start=1):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (rrf_k + rank)

    fused = sorted(scores.items(), key=lambda x: -x[1])
    return [doc_id for doc_id, _ in fused[:k]]

K = 3
rows = []
for item in GOLDEN_REAL:
    query = item["query"]
    relevant = item["relevant"]

    tfidf_retrieved = retrieve(query, k=10)
    embed_retrieved = retrieve_embed(query, k=10)
    hybrid_retrieved = retrieve_hybrid(query, k=10)

    rows.append({
        "query": query,
        "tfidf_recall": recall_at_k(tfidf_retrieved, relevant, K),
        "tfidf_mrr": mrr(tfidf_retrieved, relevant),
        "embed_recall": recall_at_k(embed_retrieved, relevant, K),
        "embed_mrr": mrr(embed_retrieved, relevant),
        "hybrid_recall": recall_at_k(hybrid_retrieved, relevant, K),
        "hybrid_mrr": mrr(hybrid_retrieved, relevant),
    })

results_hybrid = pd.DataFrame(rows)
results_hybrid


,query,tfidf_recall,tfidf_mrr,embed_recall,embed_mrr,hybrid_recall,hybrid_mrr
0,What Is An Annual Premium For Car Insurance?,0.0,0.00,1.0,1.000000,1.0,0.5
1,What Does The Medicare Suffix T Mean?,1.0,1.00,1.0,1.000000,1.0,1.0
2,Is Life Insurance An Inheritance?,1.0,1.00,1.0,1.000000,1.0,1.0
3,What Is A Good Life Insurance Policy?,0.0,0.25,0.0,0.142857,0.5,1.0
4,Do I Need Homeowners Insurance In A Condo?,0.0,0.00,0.0,0.000000,0.0,0.0
5,What Is The Average Cost Of A Whole Life Insur...,0.0,0.00,0.0,0.000000,0.0,0.0
6,Can I Get Disability Insurance If Already Preg...,1.0,1.00,1.0,0.333333,1.0,1.0
7,Who Has The Cheapest Term Life Insurance?,0.0,0.00,0.0,0.000000,0.0,0.0


In [36]:
import faiss
import numpy as np

# normalize so inner product == cosine similarity (same math we've been using all along)
doc_embeddings_norm = doc_embeddings / np.linalg.norm(doc_embeddings, axis=1, keepdims=True)
doc_embeddings_norm = doc_embeddings_norm.astype("float32")  # faiss requires float32

dim = doc_embeddings_norm.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(doc_embeddings_norm)
print("vectors in index:", index.ntotal)  # should be 27413

faiss.write_index(index, "faiss_index.bin")
print("saved to faiss_index.bin")

# reload from disk to prove persistence actually works, not just trust it
index_reloaded = faiss.read_index("faiss_index.bin")
print("reloaded, vectors in index:", index_reloaded.ntotal)

def retrieve_faiss(query, k=5):
    q_emb = model.encode([query])[0]
    q_emb = (q_emb / np.linalg.norm(q_emb)).reshape(1, -1).astype("float32")
    scores, indices = index_reloaded.search(q_emb, k)
    return [doc_ids[i] for i in indices[0]]

# regression check: same known query, should match retrieve_embed's result exactly
# (IndexFlatIP is exact search, not approximate, so this should be identical, not just similar)
test_query = decode(questions_df.iloc[0]["question_idx"], vocab)
top5_faiss = retrieve_faiss(test_query, k=5)
top5_brute = retrieve_embed(test_query, k=5)

print("\nfaiss top-5: ", top5_faiss)
print("brute top-5:", top5_brute)
print("identical?", top5_faiss == top5_brute)


vectors in index: 27413
saved to faiss_index.bin
reloaded, vectors in index: 27413

faiss top-5:  ['16164', '21000', '16745', '24808', '99']
brute top-5: ['16164', '21000', '16745', '24808', '99']
identical? True


In [37]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")  # small, standard, well-regarded

def retrieve_reranked(query, k=5, candidate_pool=20):
    candidates = retrieve_hybrid(query, k=candidate_pool)  # wide net from what we already built
    pairs = [(query, CORPUS[doc_id]) for doc_id in candidates]
    scores = reranker.predict(pairs)
    reranked = [doc_id for _, doc_id in sorted(zip(scores, candidates), reverse=True)]
    return reranked[:k]

# test directly on row 0 -- the query where hybrid demoted the true answer from rank 1 to rank 2
test_query = decode(questions_df.iloc[0]["question_idx"], vocab)
true_labels = set(questions_df.iloc[0]["groundtruth"].split())

hybrid_result = retrieve_hybrid(test_query, k=5)
reranked_result = retrieve_reranked(test_query, k=5)

print("query:", test_query)
print("true labels:    ", true_labels)
print("hybrid top-5:   ", hybrid_result)
print("reranked top-5: ", reranked_result)


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2033.32it/s, Materializing param=classifier.weight]                                    
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


query: What Happens When Term Life Insurance Is Paid Up?
true labels:     {'16164', '99', '26337'}
hybrid top-5:    ['15032', '21301', '18404', '16164', '2700']
reranked top-5:  ['99', '26337', '6406', '24808', '16164']


In [39]:
K = 3
rows = []
for item in GOLDEN_REAL:
    query = item["query"]
    relevant = item["relevant"]

    tfidf_retrieved = retrieve(query, k=10)
    embed_retrieved = retrieve_embed(query, k=10)
    hybrid_retrieved = retrieve_hybrid(query, k=10)
    reranked_retrieved = retrieve_reranked(query, k=10)

    rows.append({
        "query": query,
        "tfidf_recall": recall_at_k(tfidf_retrieved, relevant, K),
        "tfidf_mrr": mrr(tfidf_retrieved, relevant),
        "embed_recall": recall_at_k(embed_retrieved, relevant, K),
        "embed_mrr": mrr(embed_retrieved, relevant),
        "hybrid_recall": recall_at_k(hybrid_retrieved, relevant, K),
        "hybrid_mrr": mrr(hybrid_retrieved, relevant),
        "reranked_recall": recall_at_k(reranked_retrieved, relevant, K),
        "reranked_mrr": mrr(reranked_retrieved, relevant),
    })

results_final = pd.DataFrame(rows)
results_final


,query,tfidf_recall,tfidf_mrr,embed_recall,embed_mrr,hybrid_recall,hybrid_mrr,reranked_recall,reranked_mrr
0,What Is An Annual Premium For Car Insurance?,0.0,0.00,1.0,1.000000,1.0,0.5,1.0,1.0
1,What Does The Medicare Suffix T Mean?,1.0,1.00,1.0,1.000000,1.0,1.0,1.0,1.0
2,Is Life Insurance An Inheritance?,1.0,1.00,1.0,1.000000,1.0,1.0,1.0,0.5
3,What Is A Good Life Insurance Policy?,0.0,0.25,0.0,0.142857,0.5,1.0,0.5,1.0
4,Do I Need Homeowners Insurance In A Condo?,0.0,0.00,0.0,0.000000,0.0,0.0,0.0,0.0
5,What Is The Average Cost Of A Whole Life Insur...,0.0,0.00,0.0,0.000000,0.0,0.0,0.0,0.0
6,Can I Get Disability Insurance If Already Preg...,1.0,1.00,1.0,0.333333,1.0,1.0,1.0,1.0
7,Who Has The Cheapest Term Life Insurance?,0.0,0.00,0.0,0.000000,0.0,0.0,0.0,0.0


In [43]:
from dotenv import load_dotenv
load_dotenv()

import litellm

def generate_answer(query, context, max_tokens=200):
    prompt = f"""Answer the question using only the information in the context below. If the context doesn't contain the answer, say "I don't know."

Context:
{context}

Question: {query}
Answer:"""
    response = litellm.completion(
        model="openrouter/anthropic/claude-haiku-4.5",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content

# test with the TRUE correct context first, not retrieved context --
# isolates "can it generate well given good input" before compounding with retrieval errors
item = GOLDEN_REAL[1]  # "What Does The Medicare Suffix T Mean?" -- both retrievers got this right
query = item["query"]
relevant_label = list(item["relevant"])[0]
context = CORPUS[relevant_label]

answer = generate_answer(query, context)
print("query:", query)
print("\ncontext given:\n", context[:300])
print("\ngenerated answer:\n", answer)


query: What Does The Medicare Suffix T Mean?

context given:
 The Medicare suffix T means that the person is entitled to Medicare Part A but not to retirement and survivor's or Railroad Retirement insurance. It also applies to those who are entitled to receive Part A for having end stage renal disease.

generated answer:
 The Medicare suffix T means that the person is entitled to Medicare Part A but not to retirement and survivor's or Railroad Retirement insurance. It also applies to those who are entitled to receive Part A for having end stage renal disease.


In [44]:
xitem = GOLDEN_REAL[3]  # "What Is A Good Life Insurance Policy?" -- retrieval only got recall@3=0.5 here
query = item["query"]
relevant = item["relevant"]

retrieved_labels = retrieve_reranked(query, k=3)
retrieved_context = "\n\n".join(CORPUS[label] for label in retrieved_labels)

answer = generate_answer(query, retrieved_context)
print("query:", query)
print("true labels:", relevant)
print("retrieved labels:", retrieved_labels)
print("\nretrieved context (first 500 chars):\n", retrieved_context[:500])
print("\ngenerated answer:\n", answer)


query: What Does The Medicare Suffix T Mean?
true labels: {'26868'}
retrieved labels: ['26868', '4113', '13891']

retrieved context (first 500 chars):
 The Medicare suffix T means that the person is entitled to Medicare Part A but not to retirement and survivor's or Railroad Retirement insurance. It also applies to those who are entitled to receive Part A for having end stage renal disease.

Your medicare number followed by a letter code is referred to as a claim number. It tells the type of Medicare benefits for which one qualifies. The letter T stands for "Uninsured - Entitled to HIB (Part A) under deemed or renal provisions; or Fully insured

generated answer:
 Based on the context provided, the Medicare suffix T has two main meanings:

1. It indicates that the person is entitled to Medicare Part A but not to retirement and survivor's or Railroad Retirement insurance. It also applies to those entitled to receive Part A for having end stage renal disease.

2. The letter T stands for 

In [46]:
item = GOLDEN_REAL[3]
print("testing query:", item["query"])  # verify before proceeding

query = item["query"]
relevant = item["relevant"]

retrieved_labels = retrieve_reranked(query, k=3)
retrieved_context = "\n\n".join(CORPUS[label] for label in retrieved_labels)

answer = generate_answer(query, retrieved_context)
print("true labels:", relevant)
print("retrieved labels:", retrieved_labels)
print("\ngenerated answer:\n", answer)


testing query: What Is A Good Life Insurance Policy?
true labels: {'8834', '10434'}
retrieved labels: ['8834', '3394', '6035']

generated answer:
 According to the context, a good life insurance policy is one that:

1. **Is with a reputable company** (rated A or better by AM Best)
2. **Has good guarantees** (e.g., if term insurance, the premium is guaranteed not to go up for the entire term)
3. **Is with an agency that provides good service** (you should ask about this; online agents are sometimes easier to reach than local offices)
4. **Fits your budget** (you can afford to maintain the policy long-term)
5. **Covers your needs** (the coverage amount and type match your specific purposes, such as protecting your mortgage)

The context also notes that the best life insurance policy is **the one that best fits your needs at the lowest possible cost**, and that it may be a term policy, a permanent policy, or a combination of both. To find a good policy tailored to your situation, you


In [61]:
import re
import json

def llm_judge_faithfulness(answer, context, debug=False):
    prompt = f"""You are checking whether an AI-generated answer is fully supported by the given context.

Context:
{context}

Answer to check:
{answer}

Respond with ONLY a JSON object in this exact format, no other text:
{{"faithful": true or false, "unsupported_claims": ["claim 1", "claim 2", ...]}}

If every claim in the answer is supported by the context, set "faithful" to true and "unsupported_claims" to an empty list."""

    response = litellm.completion(
        model="openrouter/anthropic/claude-haiku-4.5",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300,
    )
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r"^```(?:json)?\n?", "", raw)
    raw = re.sub(r"\n?```$", "", raw)

    if debug:
        print("raw content:", repr(response.choices[0].message.content))
        print("cleaned:", repr(raw))

    return json.loads(raw)

llm_judge_faithfulness(answer, retrieved_context, debug=True)


raw content: '```json\n{"faithful": true, "unsupported_claims": []}\n```'
cleaned: '{"faithful": true, "unsupported_claims": []}'


{'faithful': True, 'unsupported_claims': []}

In [62]:
corrupted_answer = answer + "\n\nAdditionally, this policy type is fully tax-deductible for all policyholders regardless of income."

llm_judge_faithfulness(corrupted_answer, retrieved_context, debug=True)


raw content: '```json\n{\n  "faithful": false,\n  "unsupported_claims": [\n    "this policy type is fully tax-deductible for all policyholders regardless of income"\n  ]\n}\n```'
cleaned: '{\n  "faithful": false,\n  "unsupported_claims": [\n    "this policy type is fully tax-deductible for all policyholders regardless of income"\n  ]\n}'


{'faithful': False,
 'unsupported_claims': ['this policy type is fully tax-deductible for all policyholders regardless of income']}

In [63]:
K = 3
rows = []
for item in GOLDEN_REAL:
    query = item["query"]
    relevant = item["relevant"]

    retrieved_labels = retrieve_reranked(query, k=10)
    retrieved_context = "\n\n".join(CORPUS[label] for label in retrieved_labels[:3])

    gen_answer = generate_answer(query, retrieved_context)
    judge_result = llm_judge_faithfulness(gen_answer, retrieved_context)

    rows.append({
        "query": query,
        "recall@k": recall_at_k(retrieved_labels, relevant, K),
        "mrr": mrr(retrieved_labels, relevant),
        "faithful": judge_result["faithful"],
    })

full_results = pd.DataFrame(rows)
full_results


,query,recall@k,mrr,faithful
0,What Is An Annual Premium For Car Insurance?,1.0,1.0,True
1,What Does The Medicare Suffix T Mean?,1.0,1.0,False
2,Is Life Insurance An Inheritance?,1.0,0.5,True
3,What Is A Good Life Insurance Policy?,0.5,1.0,True
4,Do I Need Homeowners Insurance In A Condo?,0.0,0.0,True
5,What Is The Average Cost Of A Whole Life Insur...,0.0,0.0,True
6,Can I Get Disability Insurance If Already Preg...,1.0,1.0,False
7,Who Has The Cheapest Term Life Insurance?,0.0,0.0,True


In [ ]:
for i in [1, 6]:
    item = GOLDEN_REAL[i]
    query = item["query"]
    relevant = item["relevant"]

    retrieved_labels = retrieve_reranked(query, k=10)
    retrieved_context = "\n\n".join(CORPUS[label] for label in retrieved_labels[:3])

    gen_answer = generate_answer(query, retrieved_context)
    judge_result = llm_judge_faithfulness(gen_answer, retrieved_context)

    print(f"=== Row {i}: {query} ===")
    print("generated answer:\n", gen_answer)
    print("\njudge result:", judge_result)
    print("\n" + "="*60 + "\n")

# The judge appears to have fabricated something to flag, not caught a real issue. This is exactly the "how do you validate the judge itself" scenario from your interview prep, playing out live: we're doing that human-verification step right now, and the judge just failed it, not the generation.

# The complaint is about framing/emphasis, leading with one interpretation before mentioning the conflict, rather than surfacing the ambiguity upfront. That's arguably an answer-quality concern, not a faithfulness violation, our judge prompt only asked "is every claim supported," but it's answering a broader "did you handle this well" question instead. This is a prompt-scope ambiguity, not a clean judge error like row 1.

=== Row 1: What Does The Medicare Suffix T Mean? ===
generated answer:
 Based on the context provided, the Medicare suffix T means:

The person is entitled to Medicare Part A but not to retirement and survivor's or Railroad Retirement insurance. It also applies to those who are entitled to receive Part A for having end stage renal disease.

More specifically, the letter T stands for "Uninsured - Entitled to HIB (Part A) under deemed or renal provisions; or Fully insured who have elected entitlement only to HIB (Part A)," where HIB (Health Insurance Benefits) refers to Part A, which is the hospitalization and skilled care part of Medicare.

judge result: {'faithful': False, 'unsupported_claims': ['the person has both Medicare Parts A & B']}


=== Row 6: Can I Get Disability Insurance If Already Pregnant? ===
generated answer:
 Yes, you can get disability insurance if you are already pregnant, but with important limitations:

- Your current pregnancy will **not be covered** due to pre-ex

In [65]:
def check_regression(current, baseline, tolerance=0.05):
    failures = []
    for metric, base_val in baseline.items():
        cur_val = current[metric]
        if cur_val < base_val - tolerance:
            failures.append(f"{metric} regressed: {cur_val:.3f} < baseline {base_val:.3f} (tol {tolerance})")
    return failures

baseline = {
    "recall@k": full_results["recall@k"].mean(),
    "mrr": full_results["mrr"].mean(),
    "faithful_rate": full_results["faithful"].mean(),
}

print("Baseline (raw, from today's full real-pipeline run):")
for k, v in baseline.items():
    print(f"  {k}: {v:.3f}")

import json
with open("baseline_metrics.json", "w") as f:
    json.dump(baseline, f, indent=2)
print("\nsaved to baseline_metrics.json")

# sanity check: running the gate against itself should always pass
failures = check_regression(baseline, baseline)
print("\nself-check failures (should be empty):", failures)


Baseline (raw, from today's full real-pipeline run):
  recall@k: 0.562
  mrr: 0.562
  faithful_rate: 0.750

saved to baseline_metrics.json

self-check failures (should be empty): []


In [66]:
import json
from datetime import datetime, timezone

ONLINE_LOG_PATH = "online_eval_log.jsonl"

def process_live_query(query, k=3, log_path=ONLINE_LOG_PATH):
    retrieved_labels = retrieve_reranked(query, k=10)
    retrieved_context = "\n\n".join(CORPUS[label] for label in retrieved_labels[:k])

    gen_answer = generate_answer(query, retrieved_context)
    judge_result = llm_judge_faithfulness(gen_answer, retrieved_context)

    record = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "query": query,
        "retrieved_labels": retrieved_labels[:k],
        "answer": gen_answer,
        "faithful": judge_result["faithful"],
        "unsupported_claims": judge_result["unsupported_claims"],
    }

    with open(log_path, "a") as f:
        f.write(json.dumps(record) + "\n")

    return record

# simulate a genuinely new user query -- one we've never used or labeled before
new_query = decode(questions_df.iloc[500]["question_idx"], vocab)
print("simulated live query:", new_query)

record = process_live_query(new_query)
print("\nlogged record:")
for key, val in record.items():
    print(f"  {key}: {str(val)[:200]}")


simulated live query: Can You Have Long Term Care Insurance And Medicaid?

logged record:
  timestamp: 2026-09-04T09:18:13.292415+00:00
  query: Can You Have Long Term Care Insurance And Medicaid?
  retrieved_labels: ['4807', '19861', '13946']
  answer: Yes, you can have Long Term Care Insurance and Medicaid at the same time. When you have both, Medicaid is the secondary payer and your private Long Term Care policy is the primary payer for your long 
  faithful: True
  unsupported_claims: []


In [67]:
new_indices = [100, 250, 750, 1000, 1500]
for idx in new_indices:
    q = decode(questions_df.iloc[idx]["question_idx"], vocab)
    print(f"processing query {idx}: {q}")
    process_live_query(q)


processing query 100: How Much Does Life Insurance Cost?
processing query 250: Does my sister get the insurance policy that I carry on my mom when she passes?
processing query 750: Can I Keep My Car Without Insurance?
processing query 1000: Does Home Insurance Cover Leaking Windows?
processing query 1500: How Long Does It Take To Get Money From Home Insurance?


In [68]:
import pandas as pd

log_records = []
with open(ONLINE_LOG_PATH) as f:
    for line in f:
        log_records.append(json.loads(line))

log_df = pd.DataFrame(log_records)
print(f"total logged queries: {len(log_df)}\n")
print(log_df[["timestamp", "query", "faithful"]])

print("\noverall faithful rate:", log_df["faithful"].mean())

# rolling faithful rate -- this is the actual "drift monitoring" pattern in miniature
log_df["rolling_faithful_rate"] = log_df["faithful"].rolling(window=3, min_periods=1).mean()
print("\nrolling faithful rate (window=3):")
print(log_df[["query", "faithful", "rolling_faithful_rate"]])


total logged queries: 6

                          timestamp  \
0  2026-09-04T09:18:13.292415+00:00   
1  2026-09-04T09:22:05.626441+00:00   
2  2026-09-04T09:22:10.547493+00:00   
3  2026-09-04T09:22:14.061419+00:00   
4  2026-09-04T09:22:18.115237+00:00   
5  2026-09-04T09:22:21.463444+00:00   

                                               query  faithful  
0  Can You Have Long Term Care Insurance And Medi...      True  
1                 How Much Does Life Insurance Cost?      True  
2  Does my sister get the insurance policy that I...      True  
3               Can I Keep My Car Without Insurance?      True  
4         Does Home Insurance Cover Leaking Windows?      True  
5  How Long Does It Take To Get Money From Home I...      True  

overall faithful rate: 1.0

rolling faithful rate (window=3):
                                               query  faithful  \
0  Can You Have Long Term Care Insurance And Medi...      True   
1                 How Much Does Life Insurance Cost

In [3]:
## STEP 1 : Corpus 
CORPUS = {
    "d1": "Water damage to the kitchen ceiling caused by a burst pipe is covered under the standard homeowners policy.",
    "d2": "Flood damage from rising external water is excluded under the standard policy and requires separate flood coverage.",
    "d3": "The deductible for water damage claims under the standard policy is $500 per incident.",
    "d4": "Fire damage to the garage roof after a lightning strike is covered under the standard policy.",
    "d5": "The deductible for fire damage claims is $1000 per incident, higher than water damage.",
}

for doc_id, text in CORPUS.items():
    print(doc_id, "->", text)


d1 -> Water damage to the kitchen ceiling caused by a burst pipe is covered under the standard homeowners policy.
d2 -> Flood damage from rising external water is excluded under the standard policy and requires separate flood coverage.
d3 -> The deductible for water damage claims under the standard policy is $500 per incident.
d4 -> Fire damage to the garage roof after a lightning strike is covered under the standard policy.
d5 -> The deductible for fire damage claims is $1000 per incident, higher than water damage.


In [4]:
## STEP 2 : A Simple Retriever
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

doc_ids = list(CORPUS.keys())
vectorizer = TfidfVectorizer(stop_words="english")
doc_matrix = vectorizer.fit_transform(CORPUS.values())

def retrieve(query, k=3):
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, doc_matrix)[0]
    ranked = sims.argsort()[::-1][:k]
    return [doc_ids[i] for i in ranked]

query = "what is the deductible for water damage"
top = retrieve(query, k=3)
print("query:", query)
print("top-3:", top)
for doc_id in top:
    print(" ", doc_id, "->", CORPUS[doc_id])


query: what is the deductible for water damage
top-3: ['d5', 'd3', 'd1']
  d5 -> The deductible for fire damage claims is $1000 per incident, higher than water damage.
  d3 -> The deductible for water damage claims under the standard policy is $500 per incident.
  d1 -> Water damage to the kitchen ceiling caused by a burst pipe is covered under the standard homeowners policy.


In [6]:
## STEP 3: Golden Query Set
GOLDEN = [
    {"query": "what is the deductible for water damage",           "relevant": {"d3"}},
    {"query": "is flood damage covered under the standard policy", "relevant": {"d2"}},
    {"query": "deductible for fire damage claims",                 "relevant": {"d5"}},
]

for item in GOLDEN:
    print(item["query"], "->", item["relevant"])


what is the deductible for water damage -> {'d3'}
is flood damage covered under the standard policy -> {'d2'}
deductible for fire damage claims -> {'d5'}


In [ ]:
## STEP 4 : The Metrics
def recall_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & relevant) / len(relevant)

def precision_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & relevant) / k

def mrr(retrieved, relevant): # The rank of first relevant result
    for rank, doc in enumerate(retrieved, start=1):
        if doc in relevant:
            return 1 / rank
    return 0.0

# manual trace, using Step 2's retriever on Step 3's first query
retrieved = retrieve("what is the deductible for water damage", k=3)
relevant = {"d3"}

print("retrieved:", retrieved)
print("Recall@3   =", recall_at_k(retrieved, relevant, 3))
print("Precision@3=", precision_at_k(retrieved, relevant, 3))
print("MRR        =", mrr(retrieved, relevant))

retrieved: ['d5', 'd3', 'd1']
Recall@3   = 1.0
Precision@3= 0.3333333333333333
MRR        = 0.5


In [ ]:
import pandas as pd

rows = []
for item in GOLDEN:
    retrieved = retrieve(item["query"], k=3)
    rows.append({
        "query": item["query"],
        "recall@k": recall_at_k(retrieved, item["relevant"], 3),
        "precision@k": precision_at_k(retrieved, item["relevant"], 3),
        "mrr": mrr(retrieved, item["relevant"]),
    })

results = pd.DataFrame(rows)
results

print("Mean across all queries:")
print(results[["recall@k", "precision@k", "mrr"]].mean())

print("\nSorted by MRR (weakest first):")
print(results.sort_values("mrr")[["query", "mrr"]].to_string(index=False))



,query,recall@k,precision@k,mrr
0,what is the deductible for water damage,1.0,0.333333,0.5
1,is flood damage covered under the standard policy,1.0,0.333333,1.0
2,deductible for fire damage claims,1.0,0.333333,1.0


In [ ]:
## ONLINE METRICS
query = "what is the deductible for water damage"
context = CORPUS["d3"]  # the doc our retriever actually surfaced as relevant

faithful_answer = "The deductible for water damage claims is $500 per incident."
hallucinated_answer = "The deductible for water damage claims is $500 per incident, and it's automatically waived for customers with more than 5 years of tenure."

print("Context:", context)
print("\nFaithful answer:", faithful_answer)
print("Hallucinated answer:", hallucinated_answer)


Context: The deductible for water damage claims under the standard policy is $500 per incident.

Faithful answer: The deductible for water damage claims is $500 per incident.
Hallucinated answer: The deductible for water damage claims is $500 per incident, and it's automatically waived for customers with more than 5 years of tenure.


In [11]:
import re

def judge_faithfulness(answer, context, overlap_threshold=0.5):
    context_words = set(re.findall(r"\w+", context.lower()))
    clauses = [c.strip() for c in re.split(r"\.|,| and ", answer) if c.strip()]

    unsupported = []
    for clause in clauses:
        clause_words = set(re.findall(r"\w+", clause.lower()))
        overlap = len(clause_words & context_words) / len(clause_words)
        if overlap < overlap_threshold:
            unsupported.append(clause)

    verdict = "FAITHFUL" if not unsupported else "UNSUPPORTED CLAIMS FOUND"
    return verdict, unsupported

for label, answer in [("faithful", faithful_answer), ("hallucinated", hallucinated_answer)]:
    verdict, unsupported = judge_faithfulness(answer, context)
    print(f"{label}: {verdict}")
    if unsupported:
        print("  unsupported clause(s):", unsupported)


faithful: FAITHFUL
hallucinated: UNSUPPORTED CLAIMS FOUND
  unsupported clause(s): ["it's automatically waived for customers with more than 5 years of tenure"]


In [12]:
ANSWERS = {
    "what is the deductible for water damage": hallucinated_answer,  # from Step 6b
    "is flood damage covered under the standard policy":
        "Flood damage is excluded under the standard policy and requires separate flood coverage.",
    "deductible for fire damage claims":
        "The deductible for fire damage claims is $1000 per incident, and claims are processed within 24 hours guaranteed.",
}

for item in GOLDEN:
    q = item["query"]
    ctx = CORPUS[list(item["relevant"])[0]]  # the labeled-relevant doc's text
    verdict, unsupported = judge_faithfulness(ANSWERS[q], ctx)
    print(f"{q}\n  -> {verdict}")
    if unsupported:
        print("     unsupported:", unsupported)


what is the deductible for water damage
  -> UNSUPPORTED CLAIMS FOUND
     unsupported: ["it's automatically waived for customers with more than 5 years of tenure"]
is flood damage covered under the standard policy
  -> FAITHFUL
deductible for fire damage claims
  -> UNSUPPORTED CLAIMS FOUND
     unsupported: ['claims are processed within 24 hours guaranteed']


In [13]:
def check_regression(current, baseline, tolerance=0.05):
    failures = []
    for metric, base_val in baseline.items():
        cur_val = current[metric]
        if cur_val < base_val - tolerance:
            failures.append(f"{metric} regressed: {cur_val:.3f} < baseline {base_val:.3f} (tol {tolerance})")
    return failures

# faithfulness rate across the golden set, from Step 6c
faithful_count = sum(
    judge_faithfulness(ANSWERS[item["query"]], CORPUS[list(item["relevant"])[0]])[0] == "FAITHFUL"
    for item in GOLDEN
)
faithful_rate = faithful_count / len(GOLDEN)

current = results[["recall@k", "precision@k", "mrr"]].mean().to_dict()
current["faithful_rate"] = faithful_rate
print("Current:", current)

baseline = {"recall@k": 1.0, "precision@k": 0.333, "mrr": 0.833, "faithful_rate": 0.333}

failures = check_regression(current, baseline)
if failures:
    print("\nREGRESSION DETECTED:")
    for f in failures:
        print(" -", f)
else:
    print("\nAll metrics within tolerance of baseline. Safe to ship.")


Current: {'recall@k': 1.0, 'precision@k': 0.3333333333333333, 'mrr': 0.8333333333333334, 'faithful_rate': 0.3333333333333333}

All metrics within tolerance of baseline. Safe to ship.
